In [1]:
import csv
import firebase_admin
from firebase_admin import credentials
from firebase_admin import firestore


# Use a service account
cred = credentials.Certificate('foraging-game-6b695-firebase-adminsdk-5qlrf-1c634aa2b1.json') # **TODO** input your secret key into the certificate
firebase_admin.initialize_app(cred)
db = firestore.client()


In [48]:

def subjcsvread(subjects, csvFileName, db, collection):
    subjectList = []
    for subj in subjects:

        try:
            docs = db.collection(collection).where(u'id', u'==',subj).stream()

            for doc in docs:
                fields = doc.to_dict()
                info = (fields.get('start_time'), fields.get('id'), fields.get('age'), fields.get('clampQ'), fields.get('comments'), fields.get('currTrial'), fields.get('handedness'), fields.get('ethnicity'), fields.get('mousetype'), fields.get('race'), fields.get('returner'), fields.get('sex'), fields.get('browsertype'))
                subjectList.append(info)
        except:
            print(subj + "doesn't exist!")
            continue

    return subjectList

def getSubjectData(subjects, csvFileName, db, collection):

    subjectList = subjcsvread(subjects, csvFileName, db, collection)

    #Set up file to write to
    file = open(csvFileName, 'w')
    writer = csv.writer(file)
    header = ('StartTime', 'Subject ID', 'Age', 'Clamp Question', 'Comments', 'Completed Trials', 'Handedness', 'Ethnicity', 'Mouse Type', 'Race', 'Returner', 'Sex', 'Browsertype')
    writer.writerow(header)
    writer.writerows(subjectList)
    file.close()

def addSubjectData(subjects, csvFileName, db, collection):
    subjectList = subjcsvread(subjects, csvFileName, db, collection)

    #Set up file to write to
    file = open(csvFileName, 'a')
    writer = csv.writer(file)
    writer.writerows(subjectList)
    file.close()


def trialcsvread(collection, numTrials, csvFileName, subjects, db):
    #Create array with complete set of id's in database
    #Comment out this portion if you are not using 'id' as your field
    ids = []
    for subj in subjects:
      i=int(numTrials/1)
      for n in range(0,i):
            m=n*1
            ids.append(subj + str(m+1))
    print(ids)
    #Create array with complete set of id's in database
    trials = []
    for trialID in ids:
        # try:
        docs = db.collection(collection).where(u'id', u'==', trialID).stream()

        for doc in docs:
            print(trialID)
            fields = doc.to_dict()
            # exp_ID = fields.get('experimentID')
            name = fields.get('id')
            travelTime = fields.get('travelTime')
            block = fields.get('block');
            tree_num = fields.get('tree')
            timeStamp = fields.get('timeStamp')
            timeRemaining = fields.get('timeRemaining')
            action = fields.get('action')
            currentPresses = fields.get('currentPresses')
            requiredPresses = fields.get('requiredPresses')
            score = fields.get('score')
            keyIncRate = fields.get('keyIncRate')
            for i in range(len(tree_num)):
                # print(f"Loop Number: {i}")
                if i<1:

                    # trial = (name, tree_num, timeRemaining, action, travelTime, requiredPresses, currentPresses,
                    #         keyIncRate, score)
                    trial = (name, block[i], tree_num[i], timeRemaining[i], timeStamp[i], action[i], travelTime[i], requiredPresses[i], currentPresses[i],
                            keyIncRate[i], score[i])
                    
                    # trial = (exp_ID, name, currDate_arr[i], trialnum_arr[i], tgtAng_arr[i], trialType_arr[i], rot_arr[i],
                    # handang_arr[i], rt_arr[i], mt_arr[i], search_arr[i], reachfb_arr[i],target_dist[i], screenheight_arr[i], screenwidth_arr[i],[1],[1],[1],[1],[1],[1],[1])
                    # print(trial)
                    trials.append(trial)
                else:
                    # trial = (name, tree_num, timeRemaining, action, travelTime, requiredPresses, currentPresses,
                    #         keyIncRate, score)
                    trial = (name, block[i], tree_num[i], timeRemaining[i], timeStamp[i], action[i], travelTime[i], requiredPresses[i], currentPresses[i],
                            keyIncRate[i], score[i])
                    # trial = (exp_ID, name, currDate_arr[i], trialnum_arr[i], tgtAng_arr[i], trialType_arr[i], rot_arr[i],
                    # handang_arr[i], rt_arr[i], mt_arr[i], search_arr[i], reachfb_arr[i],target_dist[i], screenheight_arr[i], screenwidth_arr[i],hands,rs,trialnumlist,tgt_x,tgt_y,signs,mirr)
                    # print(trial)
                    trials.append(trial)
        # except:
        #     print(trialID + "wasn't completed!")
        #     # print(trial)
        #     continue

    return trials


def getTrialData(collection, numTrials, csvFileName, subjects, db):

    trials = trialcsvread(collection, numTrials, csvFileName, subjects, db)

    #Set up file to write to
    file = open(csvFileName, 'w')
    writer = csv.writer(file)
    # header = ('Experiment Name', 'Subject ID', 'Start Time', 'Trial Number', 'Target Angle', 'Cursor FB', 'Rotation', 'Hand Angle',
    # 'RT', 'MT', 'Search Time', 'Reach FB','dist', 'Screenheight', 'Screenwidth','hands','rs','trialnumlist','tgt_x','tgt_y','signs','mirror')
    header = ('Subject ID', 'Block', 'TreeNumber', 'TimeRemaining','TimeStamp', 'Action', 'TravelTime', 'RequiredPresses', 'CurrentPresses', 
              'KeyPressIncreaseRate', 'score')

    writer.writerow(header)
    writer.writerows(trials)
    file.close()

def addTrialData(collection, numTrials, csvFileName, subjects, db):

    #Create array with complete set of id's in database
    trials = trialcsvread(collection, numTrials, csvFileName, subjects, db)

    #Set up file to write to

    file = open(csvFileName, 'a')
    writer = csv.writer(file)
    emptyrow = []

    writer.writerows(trials)
    file.close()


# subjects =['5fbda54192401b1cba1129b3', '66672cf74a7ce26512636d19', '65de3fafab17ebda9cb762aa', '5e9b5a4544abb6061b193dea', '66572731cb06ce43c72bdf8a',
#           '62713a003caa3feec5c6a0ba', '66320c3fd918f178605a74f8', '6644a0c8ee3996be6fc1fc1a', '5e4e844553bc702cdf68404d', 'A80BB52EAF9C4B309F6DCA5F17D27C4F',
#           '665e50249febb72040d9a8aa', '6691399dd79826a6608cfb46', '652e11907e5db7a9f62d7258']

subjects = ['66a0fbfd52a85e69af5dd200', '5d268981c22768000166c7c7', '66a18ca34dd5ac0a73571bd8', '669d506f9b7f57ae6e07f5b6', '66a218f455df3fda0c1230c1',
           '664a1bcbac985050a4575af2', '614ce29d78ae25eced10ede1', '66412a3ce07d7c50170a3eff', '669bbcd0463407058421b400', '5589662afdf99b7416026f3a', 
              '644c34bee80856c9546c7312', '5d74001d391b6600175f433b', '5b9c2c816f92f30001ecce7a', '5e5291ff6c3d7d2a79ec869e', '6668715c876923ac0a82507b']

# subjects = ['test_db_async3']

# getSubjectData(subjects,'add_trans_ad2ep_ccw_sub.csv', db, 'Subjects')
# getTrialData('Trials', 920, 'add_trans_ad2ep_ccw.csv', subjects, db)

# getTrialData('prolificTrials',1, 'prolific_dat.csv', subjects, db)
getTrialData('prolificLongFirstTrials',1, 'prolific_dat_longfirst.csv', subjects, db)
# getTrialData('testTrials',1, 'test_db_async3.csv', subjects, db)
# getSubjectData(subjects,'mirror_test.csv', db, 'Subjects')
# getTrialData('Trials',200, 'kelsey0.csv', subjects, db)
#getTrialData('Trials', 95, 'propshift4_0July2020_a.csv', subjects, db)
#getSubjectData(subjects, 'subj_arraytest_07July2020.csv', db, 'Subjects')

['66a0fbfd52a85e69af5dd2001', '5d268981c22768000166c7c71', '66a18ca34dd5ac0a73571bd81', '669d506f9b7f57ae6e07f5b61', '66a218f455df3fda0c1230c11', '664a1bcbac985050a4575af21', '614ce29d78ae25eced10ede11', '66412a3ce07d7c50170a3eff1', '669bbcd0463407058421b4001', '5589662afdf99b7416026f3a1', '644c34bee80856c9546c73121', '5d74001d391b6600175f433b1', '5b9c2c816f92f30001ecce7a1', '5e5291ff6c3d7d2a79ec869e1', '6668715c876923ac0a82507b1']
66a0fbfd52a85e69af5dd2001
5d268981c22768000166c7c71
66a18ca34dd5ac0a73571bd81
669d506f9b7f57ae6e07f5b61
66a218f455df3fda0c1230c11
664a1bcbac985050a4575af21
614ce29d78ae25eced10ede11
66412a3ce07d7c50170a3eff1
669bbcd0463407058421b4001
5589662afdf99b7416026f3a1
644c34bee80856c9546c73121
5d74001d391b6600175f433b1
5b9c2c816f92f30001ecce7a1
5e5291ff6c3d7d2a79ec869e1
6668715c876923ac0a82507b1
